# Manual Linecut Extraction (processed_3x1um, .npz)

Scope of this notebook: load the 2D maps, flatten Z, **manually extract+verify one linecut per wavenumber** (same pattern as `response_Shizhe.ipynb`'s G-CIPS4 dispersion section -- look at the mapping, place the cut, store it, repeat), build the waterfall, align. Once that's all good, export to CSV in the same column format the old data used, so `fitting_pipeline.ipynb` / `nanoftir_Shizhe.py` can do the CHT/Hankel/FFT fitting exactly as before -- no fitting logic lives in this notebook.

All the extraction/plotting/waterfall functions live in `snippet.linecut_extraction` (shared, not GMG-specific -- the same module works for G-CIPS-style data too), so this notebook only calls them with GMG's data.

**Why per-wn manual instead of one shared geometry**: a manual horizontal align (section 6) only corrects drift *along* the cut direction. If the sample also drifted *perpendicular* to the cut between scans (vertical drift, or rotation), a single shared `(center, angle)` would silently sample the wrong slice for some wn and no amount of after-the-fact horizontal shifting fixes that. Looking at each wn's own map before extracting catches that; the horizontal align step is then only for lining up the waterfall, not for correcting sampling errors.

In [ ]:
import sys, os
sys.path.insert(0, '.')
sys.path.append('/Users/shizhe/envsetting')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loadnpz import loadnpz
from snippet.linecut_extraction import (
    suggest_flat_points, fit_plane_and_subtract, extract_and_plot,
    plot_waterfall_3channel, plot_mapping_waterfall,
)
get_ipython().run_line_magic('matplotlib', 'widget')

plt.rcParams.update({
    'font.size': 12, 'axes.linewidth': 1.5,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
})


## 1. Load all wavenumbers

In [ ]:
import glob, re

data_dir = 'data/processed_3x1um'
file_paths = glob.glob(f'{data_dir}/*_AVG.npz')

def get_wn(p):
    m = re.search(r'(\d+)cm-1', p)
    return int(m.group(1)) if m else 0

file_paths = sorted(file_paths, key=get_wn, reverse=True)
wn_list = [f"{get_wn(p)}cm-1" for p in file_paths]

scans = {wn: loadnpz(p) for wn, p in zip(wn_list, file_paths)}
print(f"Loaded {len(scans)} wavenumbers:", wn_list)
print("Map shape (ny, nx):", scans[wn_list[0]]['O3A'].shape,
      "| pixel size (nm):", scans[wn_list[0]]['x_pixelsize_nm'])


## 2. Z-plane correction

The raw `Z` in these npz files is only min-subtracted, **not** plane-corrected -- there's a real ~20-25nm tilt across the field of view (corner-to-corner spread on 860cm-1 is ~24nm, far more than a graphene step). `suggest_flat_points` proposes 3 well-separated, locally-flat spots; `fit_plane_and_subtract` averages Z in a disk around each, fits a plane through the 3 `(x, y, z_mean)` points, and subtracts it from the whole image -- this is what every `extract_and_plot` call below actually samples (pass `z_corrected[wn]`, not the raw `scans[wn]['Z']`). Override `plane_points_px[wn]` with your own `(x_px, y_px)` triplets any time; rerun this cell after editing.

In [ ]:
# Auto-suggested per-wn plane points (pixel coords). Edit any entry to override, e.g.:
#   plane_points_px['860cm-1'] = [(15, 10), (150, 55), (80, 30)]
plane_points_px = {wn: suggest_flat_points(scans[wn]['Z']) for wn in wn_list}
plane_radius_px = 4   # disk radius (px) averaged around each point

z_corrected = {}
for wn in wn_list:
    z_corr, coeffs, pts = fit_plane_and_subtract(scans[wn]['Z'], plane_points_px[wn], radius_px=plane_radius_px)
    z_corrected[wn] = z_corr

print("Plane-corrected Z for all wn. Example points used for 860cm-1:", plane_points_px['860cm-1'])


In [ ]:
# Visual check on one wn: raw Z (with suggested points marked) vs corrected Z
check_wn = '860cm-1'
z_raw = scans[check_wn]['Z']
z_corr = z_corrected[check_wn]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, img, title in zip(axes, [z_raw, z_corr], ['Z (raw, tilted)', 'Z (plane-corrected)']):
    im = ax.imshow(img, cmap='gray', origin='upper')
    fig.colorbar(im, ax=ax, shrink=0.8, label='nm')
    ax.set_title(title, fontweight='bold')
for x0, y0 in plane_points_px[check_wn]:
    axes[0].plot(x0, y0, 'o', color='red', ms=6, markeredgecolor='white')
    axes[0].add_patch(plt.Circle((x0, y0), plane_radius_px, fill=False, edgecolor='red', lw=1.2))
fig.tight_layout()
os.makedirs('figures/manual_linecut', exist_ok=True)
fig.savefig(f'figures/manual_linecut/{check_wn}_plane_correction_check.png', dpi=150, bbox_inches='tight')
fig


## 3. Per-wn manual linecut extraction

`extract_and_plot(...)` (from `snippet.linecut_extraction`) draws the O3A / O3P / Z(corrected) maps **at their true physical aspect ratio** (3x1um, not squares -- that was a bug in the previous version: it forced `aspect='auto'`, which stretches the image to fill a square-ish subplot regardless of the real extent) side by side with the proposed cut overlaid as a cyan patch, then the resulting 1D profiles in a second figure below. Look at both, adjust the geometry args, rerun the cell until the cut looks right for *that* wn, then it's appended to `linecut_store` (re-running a wn's cell overwrites only its own entry). `linecut_store` is just a plain list -- print it directly to see what's been captured so far.

Every per-wn cell below starts from the same default geometry (pulled from the npz's own `lineprofile` metadata -- verified to reproduce the old CSV's O3A values to float32 precision for 860cm-1), as a reasonable starting point you adjust from. The measured edge x-position at that default row, per wn, for reference while you place each cut:

```
1000cm-1: 34px   991cm-1: 30px   980cm-1: 28px   970cm-1: 27px   960cm-1: 23px
 950cm-1: 24px    941cm-1: 24px   930cm-1: 24px   920cm-1: 26px   911cm-1: 39px
 900cm-1: 31px    890cm-1: 33px   880cm-1: 36px   870cm-1: 36px   860cm-1: 38px
```

All `extract_and_plot` keyword args, for reference (every one of them can be passed in the per-wn cells below if you need to override beyond the geometry):

```
center_um, radius_um, angle_deg          -- cut geometry (required)
extract_mode='rect'|'wedge'              -- parallel strip (fixed width) vs angular sector (widens with distance)
rect_width_um / angle_width_deg          -- width of the cut, meaning depends on extract_mode
backward_ext_um                          -- also sample this far *behind* center_um (negative distance)
cmap_amp/cmap_phase/cmap_z               -- map colors; cmap_amp=None -> Sky
amp_range/phase_range/z_range            -- (vmin, vmax) per map, or None for auto
lc_color, lc_alpha                       -- cut overlay patch color/opacity
show_maps, show_profiles                 -- turn either figure off if you don't need it
show_align_marker                        -- dashed line + shading at distance=0 in the profile figure
store, append_to_store, tag              -- which list to append to (defaults to linecut_store below)
```

In [ ]:
linecut_store = []


In [ ]:
# Shared starting geometry for every per-wn cell below (from the npz's own line-profile
# metadata -- each cell can freely override center_um/angle_deg/radius_um/rect_width_um).
_ref = scans[wn_list[0]]
_lp = _ref['info']['lineprofile']['profiles'][0]
_px_x = _ref['x_pixelsize_nm'] / 1000.0
_px_y = _ref['y_pixelsize_nm'] / 1000.0
_x0px, _y0px = _lp['native_start_px']
_x1px, _y1px = _lp['native_end_px']

DEFAULT_CENTER_UM = (_x0px * _px_x, _y0px * _px_y)
DEFAULT_ANGLE_DEG = float(np.degrees(np.arctan2((_y1px - _y0px) * _px_y, (_x1px - _x0px) * _px_x)))
DEFAULT_RADIUS_UM = float(np.hypot((_x1px - _x0px) * _px_x, (_y1px - _y0px) * _px_y))
DEFAULT_WIDTH_UM = _lp['width_px'] * _px_x
print("DEFAULT_CENTER_UM", DEFAULT_CENTER_UM, "DEFAULT_ANGLE_DEG", DEFAULT_ANGLE_DEG,
      "DEFAULT_RADIUS_UM", DEFAULT_RADIUS_UM, "DEFAULT_WIDTH_UM", DEFAULT_WIDTH_UM)


#### 1000cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['1000cm-1']['O3A'], scans['1000cm-1']['O3P'], z_corrected['1000cm-1'],
    pixelsize_um  = scans['1000cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '1000cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 991cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['991cm-1']['O3A'], scans['991cm-1']['O3P'], z_corrected['991cm-1'],
    pixelsize_um  = scans['991cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '991cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 980cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['980cm-1']['O3A'], scans['980cm-1']['O3P'], z_corrected['980cm-1'],
    pixelsize_um  = scans['980cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '980cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 970cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['970cm-1']['O3A'], scans['970cm-1']['O3P'], z_corrected['970cm-1'],
    pixelsize_um  = scans['970cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '970cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 960cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['960cm-1']['O3A'], scans['960cm-1']['O3P'], z_corrected['960cm-1'],
    pixelsize_um  = scans['960cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '960cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 950cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['950cm-1']['O3A'], scans['950cm-1']['O3P'], z_corrected['950cm-1'],
    pixelsize_um  = scans['950cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '950cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 941cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['941cm-1']['O3A'], scans['941cm-1']['O3P'], z_corrected['941cm-1'],
    pixelsize_um  = scans['941cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '941cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 930cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['930cm-1']['O3A'], scans['930cm-1']['O3P'], z_corrected['930cm-1'],
    pixelsize_um  = scans['930cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '930cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 920cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['920cm-1']['O3A'], scans['920cm-1']['O3P'], z_corrected['920cm-1'],
    pixelsize_um  = scans['920cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '920cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 911cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['911cm-1']['O3A'], scans['911cm-1']['O3P'], z_corrected['911cm-1'],
    pixelsize_um  = scans['911cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '911cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 900cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['900cm-1']['O3A'], scans['900cm-1']['O3P'], z_corrected['900cm-1'],
    pixelsize_um  = scans['900cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '900cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 890cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['890cm-1']['O3A'], scans['890cm-1']['O3P'], z_corrected['890cm-1'],
    pixelsize_um  = scans['890cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '890cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 880cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['880cm-1']['O3A'], scans['880cm-1']['O3P'], z_corrected['880cm-1'],
    pixelsize_um  = scans['880cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '880cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 870cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['870cm-1']['O3A'], scans['870cm-1']['O3P'], z_corrected['870cm-1'],
    pixelsize_um  = scans['870cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '870cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


#### 860cm-1

In [ ]:
entry, linecut_store = extract_and_plot(
    scans['860cm-1']['O3A'], scans['860cm-1']['O3P'], z_corrected['860cm-1'],
    pixelsize_um  = scans['860cm-1']['x_pixelsize_nm'] / 1000.0,
    wn            = '860cm-1',
    center_um     = DEFAULT_CENTER_UM,   # (x, y) in um -- move along/across the edge as needed
    angle_deg     = DEFAULT_ANGLE_DEG,   # 0 = horizontal; rotate if the edge isn't parallel to the default cut
    radius_um     = DEFAULT_RADIUS_UM,   # how far the cut extends
    rect_width_um = DEFAULT_WIDTH_UM,    # averaging width of the cut
    store         = linecut_store,        # stores into linecut_store once this looks right
)


## 4. Waterfall (unaligned) -- visual check before aligning

Two styles available, both from `snippet.linecut_extraction`:
- `plot_waterfall_3channel`: amp/phase/z each get their own panel side by side (separate `stack_offset_*` per channel since they live on very different scales) -- good for checking all three channels at once.
- `plot_mapping_waterfall`: one channel, with a 2D map + the cut location on top, sharing the x-axis -- good for the final publication-style figure (see section 4b).

In [ ]:
print(f"linecut_store has {len(linecut_store)} entries:",
      sorted([e['wn'] for e in linecut_store], key=get_wn))

fig = plot_waterfall_3channel(
    linecut_store,
    distance_key      = 'distance_nm',
    stack_offset_amp  = 0.8,    # vertical spacing for the amp panel -- raise if curves overlap
    stack_offset_pha  = 0.8,    # same, for phase
    stack_offset_z    = 2.0,    # same, for z (different scale, usually needs a bigger offset)
    normalize         = 'mean', # 'none' | 'mean' | 'minmax' | 'zscore' -- per-curve normalization before stacking
    enhance_scale     = 1.0,    # multiply each normalized curve before stacking (exaggerate features)
    cmap_name         = 'turbo',# color ramp across wn
    xlim              = None,   # e.g. (-200, 800) to zoom into the edge region
)
fig


## 5. Manual alignment

Look at the unaligned waterfall above, decide what distance_nm each wn's edge sits at, fill it in below (same `{wn: value}` convention as `fitting_pipeline.ipynb`'s `align_dict`), then re-plot.

In [ ]:
align_dict_manual = {wn: 0.0 for wn in wn_list}
# e.g. align_dict_manual['860cm-1'] = 60.0

for e in linecut_store:
    e['distance_nm_aligned'] = e['distance_nm'] - align_dict_manual[e['wn']]

fig = plot_waterfall_3channel(linecut_store, distance_key='distance_nm_aligned',
                               stack_offset_amp=0.8, stack_offset_pha=0.8, stack_offset_z=2.0)
fig


### 4b/5b. Mapping + single-channel waterfall (optional, publication style)

In [ ]:
_map_wn = '860cm-1'   # which wn's map to show on top -- change to any entry in wn_list
_map_channel = 'amp'  # 'amp' (Sky) | 'phase' (bwr_r) | 'z' (gray)
_scan = scans[_map_wn]
_px_x_um = _scan['x_pixelsize_nm'] / 1000.0
_px_y_um = _scan['y_pixelsize_nm'] / 1000.0
_map_image = {'amp': _scan['O3A'], 'phase': _scan['O3P'], 'z': z_corrected[_map_wn]}[_map_channel]
_map_extent_um = [0, _map_image.shape[1] * _px_x_um, _map_image.shape[0] * _px_y_um, 0]
_map_geom = dict(center_um=DEFAULT_CENTER_UM, angle_deg=DEFAULT_ANGLE_DEG,
                  radius_um=DEFAULT_RADIUS_UM, rect_width_um=DEFAULT_WIDTH_UM)

fig = plot_mapping_waterfall(
    linecut_store, _map_image, _map_extent_um, _map_geom,
    channel        = _map_channel,    # waterfall (bottom) channel -- usually matches map_channel but doesn't have to
    distance_key   = 'distance_nm_aligned',
    cmap_wf        = 'coolwarm_r',
    stack_offset   = 1.0,
    normalize      = 'mean',
    show_align     = True,
)
fig.savefig('figures/manual_linecut/mapping_waterfall_amp.png', dpi=200, bbox_inches='tight')
fig


## 6. Export to CSV (same column format as the old data)

Once the aligned waterfall looks right, this writes one CSV per wn into `data/graphene_3x1_manual/`, columns `distance_nm, O3A, O3P, Z_nm, Z_nm_corrected` -- same shape as the original `data/graphene_3x1/*_AVG_lp1.csv`. Point `fitting_pipeline.ipynb`'s `data_dir` at this new folder (or change `load_aligned_wn_signal`'s `data_dir` default in `nanoftir_Shizhe.py`) and the existing CHT/Hankel/FFT fitting runs unchanged -- nothing in this notebook touches that logic.

In [ ]:
out_dir = 'data/graphene_3x1_manual'
os.makedirs(out_dir, exist_ok=True)

for e in linecut_store:
    df = pd.DataFrame({
        'distance_nm': e['distance_nm'],   # raw (unaligned) distance, matching the old CSVs'
                                            # convention -- alignment happens downstream via align_dict
        'O3A': e['amp'],
        'O3P': e['phase'],
        'Z_nm': e['z'],
        'Z_nm_corrected': e['z'] - np.nanmin(e['z']),
    })
    out_path = f"{out_dir}/{e['wn']}_AVG_lp1.csv"
    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path} ({len(df)} rows)")
